In [1]:
from matplotlib import pyplot as plt
import os
import sys
import numpy as np
import pandas as pd
import cv2

model_data = {
    'tasks': ['Flattening'],
    'method': 'diffusion',
    'fabrics': ['dress', 'longsleeve', 'pants', 'skirt'],
    'heatmap': False,
    'input_obs_rgb': False,
    'input_obs_depth': False,
    'denoise_obs': False,
    'grasp_img': False,
    'exp_dirs': [
        #'flattening-agent-cloth-mask-centre-multiple-prioritise-middle+RGB2M-on-mix-800k-all-garment-bonus-penalty-v2-50000'
        #'flattening-agent-cloth-mask-centre-multiple-prioritise-middle+RGB2M-on-mix-200k-sf-50000',
        '../test_results/real-world/human/flattening',
        '../test_results/real-world/LaGarNet/flattening',
        '../test_results/real-world/PlaNet-ClothPick/flattening',
        # '../test_results/real-world/mask-biased/flattening',
    ]
}
model = model_data
exp_dirs = model['exp_dirs']
tasks = model['tasks']
# method = model['method']
  
fabrics = model['fabrics']
heatmap = model['heatmap']
input_obs_rgb = model['input_obs_rgb']
input_obs_depth = model.get('input_obs_depth', False)
denoise_obs = model.get('denoise_obs', False)
grasp_img = model.get('grasp_img', False)

In [2]:
## get all direcotry not recursive in the current directory
def get_all_dir(path):
    print(path)
    ret = []
    for d in os.listdir(path):
        
        dir_path = os.path.join(path, d)
        if os.path.isdir(dir_path):
            ret.append(dir_path)

    return ret

# for exp_dir in exp_dirs:
#     experiments = get_all_dir(exp_dir)
#     experiments = sorted(experiments)
import json

def coverage_alignment_reward(NC1, NC2, IoU1, IoU2):
    dNC = NC2 - NC1
    du = IoU2 - IoU1
    alpha = 2
    beta = 1
    rca = max(np.tanh(alpha*dNC + du*beta), 0)
    b = 0.7

    if NC1 > 0.9 and NC2 < 0.9:
        return 0
    
    if NC2 >= 0.95:
        return b
    
    return rca


print_reward = True
OBJECT = 'garment'

In [3]:
## show action_images as a trajecotry in an experiment

def show_trajectory(experiment, plt_steps=None, task='Flattening'):
    print('experiment:', experiment)
    ## get all steps in the experiment, they are folders
    steps = [d for d in os.listdir('./' + experiment) if os.path.isdir('./' + experiment + '/' + d)]
    #print('steps:', steps)
    steps = [int(s.split('_')[-1]) for s in steps if 'step' in s]
    plt_steps = steps if plt_steps is None else plt_steps
    plt_steps = sorted(plt_steps)
    horizon = len(plt_steps)
    #print('horizon:', horizon)
   


    action_images = []
    rgb_images = []
    pick_heat_images = []
    place_heat_images = []
    mask_images = []
    depth_images = []
    input_obs_images = []
    input_obs_d_images = []
    normalised_coverages = []
    grasp_imgs = []
    denoise_obs_imgs = []
    IoUs = []
    print('plt_steps:', plt_steps)

    for i in plt_steps:
        print('step', i)
        rgb_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/rgb.png'))
        mask_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/mask.png'))
        depth_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/depth.png'))

        # it is in evaluation.json file
        
        with open(experiment + '/' + f'step_{str(i)}' + '/evaluation.json') as f:
            eval = json.load(f)
            if task == 'Flattening':
                normalised_coverages.append(eval['normalised_coverage'])
                IoUs.append(eval['max_IoU'])

        with open(experiment  + '/info.json') as f:
            info = json.load(f)
            posterior_rewards = info['posterior_rewards']
            prior_rewards = info['prior_rewards']
        
        
        if i != plt_steps[-1]:
            if input_obs_rgb:
                input_obs_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/input_obs_rgb.png'))
            
            if input_obs_depth:
                input_obs_d_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/input_obs_depth.png'))

            if denoise_obs:
                denoise_obs_imgs.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/noise_actions_obs_depth.png'))
            
            if grasp_img:
                grasp_imgs.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/grasp_image.png'))

            #action_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/action_obs_rgb.png'))
            action_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/action_image.png'))
            if heatmap:
                pick_heat_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/pick-heat.png'))
                place_heat_images.append(plt.imread(experiment + '/' + f'step_{str(i)}' + '/place-heat.png'))
    
    row = 6 if heatmap else 4
    if input_obs_rgb:
        row += 1
    if input_obs_depth:
        row += 1
    if denoise_obs:
        row += 1
    if grasp_img:
        row += 1
    depth_list = []
    for i, depth in enumerate(input_obs_d_images):
        d = cv2.applyColorMap(np.uint8(255 * depth), cv2.COLORMAP_JET)
        d = cv2.cvtColor(d, cv2.COLOR_BGR2RGB)
        depth_list.append(d)
    input_obs_d_images = depth_list

    plt_horizon = max(10, horizon)
    plt_horizon = horizon #min(plt_horizon, 20)
    fig, axs = plt.subplots(row, horizon, figsize=(plt_horizon*4, (4)*row))
    for i in range(horizon):
        axs[0, i].imshow(rgb_images[i])
        axs[1, i].imshow(mask_images[i])
       

        axs[2, i].imshow(1.0-depth_images[i], cmap='cividis')
        
        if i < horizon-1:
            axs[-1, i].imshow(action_images[i])
            b = 0

            
            
            if grasp_img:
               
                axs[-2-b, i].imshow(grasp_imgs[i])
                b += 1
            
            if denoise_obs:
                
                axs[-2-b, i].imshow(denoise_obs_imgs[i])
                b += 1

            if input_obs_rgb:
                axs[-2-b, i].imshow(input_obs_images[i])
                b += 1
            if input_obs_depth:
                #axs[-2-b, i].imshow(input_obs_d_images[i])
                ## colormap fancy
                axs[-2-b, i].imshow(input_obs_d_images[i], cmap='cividis')
                b += 1
            
            

            if heatmap:
                axs[-2-b, i].imshow(pick_heat_images[i])
                axs[-3-b, i].imshow(place_heat_images[i])
        else:
            axs[-1, i].imshow(rgb_images[-1])
        
        
        axs[0, i].axis('off')
        axs[1, i].axis('off')
        axs[2, i].axis('off')
        axs[-1, i].axis('off')
        b = 0
        print('denoise_obs:', denoise_obs)
        if grasp_img:
            axs[-2-b, i].axis('off')
            b +=1
        if denoise_obs:
            axs[-2-b, i].axis('off')
            b +=1
        

        if input_obs_rgb:
            axs[-2-b, i].axis('off')
            b +=1   
        if input_obs_depth:
            axs[-2-b, i].axis('off')
            b +=1
       

        if heatmap:
            axs[-2-b, i].axis('off')
            axs[-3-b, i].axis('off')
    
    # Add text on the first image of each row
    label_fontsize = 18
    label_color = 'white'  # Change to white for better visibility on dark images
    label_bgcolor = 'black'  # Background color for contrast
    label_alpha = 0.7
    label_pad = 5

    def add_text(ax, text, pos='bottom'):
        if pos == 'bottom':
            ax.text(0.5, 0.05, text, fontsize=label_fontsize, color=label_color,
                    transform=ax.transAxes, va='bottom', ha='center',
                    bbox=dict(facecolor=label_bgcolor, alpha=label_alpha, edgecolor='none', pad=label_pad))
        else:
            ax.text(0.5, 0.95, text, fontsize=label_fontsize, color=label_color,
                    transform=ax.transAxes, va='top', ha='center',
                    bbox=dict(facecolor=label_bgcolor, alpha=label_alpha, edgecolor='none', pad=label_pad))

    add_text(axs[0, 0], 'RGB')
    add_text(axs[1, 0], 'Mask')
    add_text(axs[2, 0], 'Depth')
    #add_text(axs[-1, 0], 'Action on Input RGB')
    for i in range(horizon):
        if task == 'Flattening':
            string2print = f'NC: {int(normalised_coverages[i]*100)}'
            if OBJECT == 'garment':
                string2print += f', IoU: {int(IoUs[i]*100)}'
            
            if print_reward == True and i > 0 and i < horizon:
                reward = coverage_alignment_reward(
                    normalised_coverages[i-1], 
                    normalised_coverages[i],
                    IoUs[i-1],
                    IoUs[i])
                string2print += f'\n GT-R: {int(reward*100)}, PST-R: {int(posterior_rewards[i]*100)}, PR-R: {int(prior_rewards[i-1]*100)},'

            add_text(axs[-1, i], string2print, pos='bottom')
        else:
            add_text(axs[-1, i], f'IoU: {IoUs[i]:.2f}', pos='bottom')
    

    b = 0

    
    if grasp_img:
        add_text(axs[-2-b, 0], 'Grasping')
        b +=1

    if denoise_obs:
        add_text(axs[-2-b, 0], 'Denoising')
        b +=1

    if input_obs_rgb:
        add_text(axs[-2-b, 0], 'Input RGB')
        
        b +=1
    if input_obs_depth:
        add_text(axs[-2-b, 0], 'Input Depth')
        b +=1
   
    if heatmap:
        add_text(axs[-2-b, 0], 'Pick Heat')
        add_text(axs[-3-b, 0], 'Place Heat')
        
    
    ## remove the inter-space between subplots
    #plt.subplots_adjust(wspace=0, hspace=0)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
    fig.tight_layout(pad=0)

    # make it tight, column inter-space is 0
    #plt.tight_layout()
    plt.savefig(experiment + '/trajectory.png')
    plt.show()



In [4]:
## Get step-wise evaluation results
import json
def get_stepwise_eval(experiment):
    steps = [d for d in os.listdir('./' + experiment) if os.path.isdir('./' + experiment + '/' + d)]
    ## filter the steps that have `goal` in the name
    steps = [s for s in steps if 'step' in s]
    #print('steps:', steps)
    horizon = len(steps)
    #print('horizon:', horizon)
    evaluations = {
        'trial': [],
        'fabric': [],
        'state': [],
    }
    
    for i in range(horizon):
        eval_file = experiment + '/' + f'step_{str(i)}' + '/evaluation.json'
        if not os.path.exists(eval_file):
            print('evaluation file not found:', eval_file)
            continue
        evaluations['fabric'].append(experiment.split('/')[-1].split('-')[0])
        evaluations['trial'].append(experiment.split('/')[-1].split('-')[-1])
        evaluations['state'].append(i)
        with open(eval_file, 'r') as f:
            eval_data = json.load(f)
            for k, v in eval_data.items():
                if k not in evaluations:
                    evaluations[k] = []
                evaluations[k].append(v)
    df = pd.DataFrame(evaluations)
    df['steps'] = horizon - 1
    #display(df)
    return df

task='flattening'

total_states=[]
total_objects_states = []

for exps in exp_dirs:
    print('############################################\n\n')
    print(exps)

    totoal_stats_IoU = []
    total_auto_success = []
    total_stats_nc = []
    total_stats_ni = []
    total_stats_s2f = []

    experiments = get_all_dir(exps)
    evals = []
    for exp in experiments:
        evals.append(get_stepwise_eval(exp))

    eval_df = pd.concat(evals)

    

    #display(eval_df)

    # Sort by trial, fabric, and state (assuming state is sortable and increasing)
    df_sorted = eval_df.sort_values(['trial', 'fabric', 'state'])

    # Define a function to apply to each group
    def get_first_or_last(group, nc_treshold, iou_threshold):
        # Find rows that meet both conditions
        filtered = group[(group['normalised_coverage'] > nc_treshold) & (group['max_IoU'] > iou_threshold)]
        if not filtered.empty:
            # If any row meets the criteria, return the first one
            return filtered.iloc[0]
        else:
            # Otherwise, return the last row in the group
            return group.iloc[-1]
    
    def get_first_or_max_nc(group, nc_threshold, iou_threshold):
        # Find rows that meet both conditions
        filtered = group[(group['normalised_coverage'] > nc_threshold) & (group['max_IoU'] > iou_threshold)]
        if not filtered.empty:
            # If any row meets the criteria, return the first one
            return filtered.iloc[0]
        else:
            # Otherwise, return the row with the maximum 'normalised_coverage'
            return group.loc[group['normalised_coverage'].idxmax()]
    
    def get_first_or_max_iou(group, nc_threshold, iou_threshold):
        # Find rows that meet both conditions
        filtered = group[(group['normalised_coverage'] > nc_threshold) & (group['max_IoU'] > iou_threshold)]
        if not filtered.empty:
            # If any row meets the criteria, return the first one
            return filtered.iloc[0]
        else:
            # Otherwise, return the row with the maximum 'normalised_coverage'
            return group.loc[group['max_IoU'].idxmax()]

    # Apply the function to each (trial, fabric) group
    result = df_sorted.groupby(['trial', 'fabric'], group_keys=False).apply(
        lambda x: get_first_or_last(x, 0.9, 0.8)
    ).reset_index(drop=True)
    result.loc[
    (result['normalised_coverage'] > 0.9) & (result['max_IoU'] > 0.8),
    'auto success'
    ] = True

    result_ = df_sorted.groupby(['trial', 'fabric'], group_keys=False).apply(
        lambda x: get_first_or_max_nc(x, 0.9, 0)
    ).reset_index(drop=True)
    result_.loc[
    (result_['normalised_coverage'] > 0.9) & (result_['max_IoU'] > 0),
    'auto success nc'
    ] = True
    result_.loc[
    ~((result_['normalised_coverage'] > 0.9) & (result_['max_IoU'] > 0)),
    'auto success nc'
    ] = False
    #display(result_)

    result_1 = df_sorted.groupby(['trial', 'fabric'], group_keys=False).apply(
        lambda x: get_first_or_max_nc(x, 0.99, 0)
    ).reset_index(drop=True)

    result_2 = df_sorted.groupby(['trial', 'fabric'], group_keys=False).apply(
        lambda x: get_first_or_max_iou(x, 0, 0.99)
    ).reset_index(drop=True)

    #display(result)
    eval_df = result_1
    eval_df['auto success'] = result['auto success']
    eval_df['auto success nc'] = result_['auto success nc']
    eval_df['steps'] = result['steps']
    eval_df['max_IoU'] = result_2['max_IoU']


    if 'max_IoU' in eval_df.columns:
        # IoU x 100
        eval_df['max_IoU'] = eval_df['max_IoU'] * 100
        #display(eval_df)

        # for each fabric trial, get the largest IoU

        IoU_stats_df = eval_df.groupby(['fabric', 'trial']).agg({'max_IoU': 'max'}).reset_index()
        totoal_stats_IoU.append(IoU_stats_df)
        # for each fabric print the mena and std of IoU from IoU_stats_df
        IoU_stats_df = IoU_stats_df.groupby('fabric').agg({'max_IoU': ['mean', 'std']}).reset_index()

       

    if 'normalised_coverage' in eval_df.columns:
        eval_df['normalised_coverage'] = eval_df['normalised_coverage'] * 100
        # get the row for the last state for eah fabric trial, and get the mean and std of normalised_coverage for each fabric
        nc_stats_df = eval_df[eval_df['normalised_coverage'] < 101.0]
        nc_stats_df = nc_stats_df.groupby(['fabric', 'trial']).agg({'normalised_coverage': 'max'}).reset_index()
        total_stats_nc.append(nc_stats_df)
        nc_stats_df = nc_stats_df.groupby('fabric').agg({'normalised_coverage': ['mean', 'std']}).reset_index()

 

    if 'normalised_improvement' in eval_df.columns:
        eval_df['normalised_improvement'] = eval_df['normalised_improvement'] * 100
        ni_stats_df = eval_df[eval_df['normalised_improvement'] < 101.0]
        ni_stats_df = ni_stats_df.groupby(['fabric', 'trial']).agg({'normalised_improvement': 'max'}).reset_index()
        total_stats_ni.append(ni_stats_df)
        ni_stats_df = ni_stats_df.groupby('fabric').agg({'normalised_improvement': ['mean', 'std']}).reset_index()


    if 'steps' in eval_df.columns:
        #eval_df['normalised_improvement'] = eval_df['normalised_improvement'] * 100
        
        s2f_stats_df = eval_df.groupby(['fabric', 'trial']).agg({'steps': 'min'}).reset_index()
        total_stats_s2f.append(s2f_stats_df)
        s2f_stats_df = s2f_stats_df.groupby('fabric').agg({'steps': ['mean', 'std']}).reset_index()

    # get the largest 'state' for each fabric trial
    auto_success_df = eval_df.groupby(['fabric', 'trial']).agg({'auto success': 'max', 'auto success nc': 'max'}).reset_index()
    # count the number of True for each fabric
    auto_success_df = auto_success_df.groupby('fabric').agg({'auto success': 'sum', 'auto success nc': 'sum'}).reset_index()

    # auto_success_nc_df = eval_df.groupby(['fabric', 'trial']).agg({'auto success nc': 'max'}).reset_index()
    # # count the number of True for each fabric
    # auto_success_nc_df = auto_success_nc_df.groupby('fabric').agg({'auto success nc': 'sum'}).reset_index()


    #display(human_success_df)
    # get the number of trials for each fabric
    num_trials_df = eval_df.groupby('fabric').agg({'trial': 'nunique'}).reset_index()

    # combine the two dataframes
    auto_success_df = auto_success_df.merge(num_trials_df, on='fabric')
    auto_success_df.columns = ['fabric', 'num_auto_success', 'num_auto_success_nc', 'num_trials']
    #display(auto_success_df)
    total_auto_success.append(auto_success_df)
    #display(auto_success_df)


    ## get total stats


    import pandas as pd

    # Concatenate each list of DataFrames if not empty
    if len(total_stats_nc) > 0:
        total_stats_nc_df = pd.concat(total_stats_nc)
    else:
        total_stats_nc_df = pd.DataFrame()

    if len(total_stats_ni) > 0:
        total_stats_ni_df = pd.concat(total_stats_ni)
    else:
        total_stats_ni_df = pd.DataFrame()

    if len(totoal_stats_IoU) > 0:
        totoal_stats_IoU_df = pd.concat(totoal_stats_IoU)
    else:
        totoal_stats_IoU_df = pd.DataFrame()

    if len(total_stats_s2f) > 0:
        total_stats_s2f_df = pd.concat(total_stats_s2f)
    else:
        total_stats_s2f_df = pd.DataFrame()

    if len(total_auto_success) > 0:
        total_auto_success_df = pd.concat(total_auto_success)
    else:
        total_auto_success_df = pd.DataFrame()

    #display(total_stats_nc_df)

    # Create a combined dataframe with aggregated statistics
    combined_total_stats = pd.DataFrame()

    if not total_stats_nc_df.empty:
        combined_total_stats['normalised_coverage_mean'] = [total_stats_nc_df['normalised_coverage'].mean()]
        combined_total_stats['normalised_coverage_std'] = [total_stats_nc_df['normalised_coverage'].std()]

    if not total_stats_ni_df.empty:
        combined_total_stats['normalised_improvement_mean'] = [total_stats_ni_df['normalised_improvement'].mean()]
        combined_total_stats['normalised_improvement_std'] = [total_stats_ni_df['normalised_improvement'].std()]

    if not totoal_stats_IoU_df.empty:
        combined_total_stats['max_IoU_mean'] = [totoal_stats_IoU_df['max_IoU'].mean()]
        combined_total_stats['max_IoU_std'] = [totoal_stats_IoU_df['max_IoU'].std()]

    if not total_stats_s2f_df.empty:
        combined_total_stats['steps_mean'] = [total_stats_s2f_df['steps'].mean()]
        combined_total_stats['steps_std'] = [total_stats_s2f_df['steps'].std()]

    if not total_auto_success_df.empty:
        combined_total_stats['num_auto_success_nc'] = [total_auto_success_df['num_auto_success_nc'].sum()]
        combined_total_stats['num_success_sum'] = [total_auto_success_df['num_auto_success'].sum()]
        combined_total_stats['num_trials_sum'] = [total_auto_success_df['num_trials'].sum()]

    combined_total_stats['agent'] = exps.split('/')[-2]
    #display(combined_total_stats)
    total_states.append(combined_total_stats)

    #display(total_stats_nc)
    
    if len(total_stats_nc) > 0:
        total_stats_nc_df = pd.concat(total_stats_nc)
        nc_agg = total_stats_nc_df.groupby('fabric')['normalised_coverage'].agg(['mean', 'std']).reset_index()
        #display(nc_agg)

    if len(total_stats_ni) > 0:
        total_stats_ni_df = pd.concat(total_stats_ni)
        ni_agg = total_stats_ni_df.groupby('fabric')['normalised_improvement'].agg(['mean', 'std']).reset_index()

    if len(totoal_stats_IoU) > 0:
        total_stats_IoU_df = pd.concat(totoal_stats_IoU)
        iou_agg = total_stats_IoU_df.groupby('fabric')['max_IoU'].agg(['mean', 'std']).reset_index()

    if len(total_stats_s2f) > 0:
        total_stats_s2f_df = pd.concat(total_stats_s2f)
        s2f_agg = total_stats_s2f_df.groupby('fabric')['steps'].agg(['mean', 'std']).reset_index()

    if len(total_auto_success) > 0:
        total_auto_success_df = pd.concat(total_auto_success)
        human_agg = total_auto_success_df.groupby('fabric')[['num_auto_success', 'num_auto_success_nc', 'num_trials']].sum().reset_index()

    #display(nc_agg)

    nc_agg = nc_agg.rename(columns={'mean': 'normalised_coverage_mean', 'std': 'normalised_coverage_std'})
    ni_agg = ni_agg.rename(columns={'mean': 'normalised_improvement_mean', 'std': 'normalised_improvement_std'})
    iou_agg = iou_agg.rename(columns={'mean': 'max_IoU_mean', 'std': 'max_IoU_std'})
    s2f_agg = s2f_agg.rename(columns={'mean': 'steps_mean', 'std': 'steps_std'})
    # human_agg likely already has clear names

    # Now merge as before
    from functools import reduce

    dfs = [df for df in [nc_agg, ni_agg, iou_agg, s2f_agg, human_agg] if 'df' in locals() and df is not None]
    summary_df = reduce(lambda left, right: pd.merge(left, right, on='fabric', how='outer'), dfs)
    summary_df['agent'] =  exps.split('/')[-2]

    total_objects_states.append(summary_df)

    #display(summary_df)

total_states = pd.concat(total_states)
total_objects_states = pd.concat(total_objects_states)
agent_rename = {
    'human': 'Human Policy',
    'LaGarNet': 'LaGarNet',
    'mask-biased': 'Mask-biased Random',
    'PlaNet-ClothPick': 'PlaNet-ClothPick'
}

fabric_rename = {
    'dress': 'Dress',
    'longsleeve': 'Longsleeve',
    'pants': 'Trousers',
    'skirt': 'Skirt' 
}

total_objects_states['agent'] = total_objects_states['agent'].replace(agent_rename)
total_states['agent'] = total_states['agent'].replace(agent_rename)

total_objects_states['fabric'] = total_objects_states['fabric'].replace(fabric_rename)



display(total_states)


display(total_objects_states)
    # total_human_success = pd.concat(total_human_success)
    # print('\n\nTotal human success')
    # display(total_human_success.agg({'num_human_success': 'sum', 'num_trials': 'sum'}))


############################################


../test_results/real-world/human/flattening
../test_results/real-world/human/flattening
############################################


../test_results/real-world/LaGarNet/flattening
../test_results/real-world/LaGarNet/flattening
############################################


../test_results/real-world/PlaNet-ClothPick/flattening
../test_results/real-world/PlaNet-ClothPick/flattening


,normalised_coverage_mean,normalised_coverage_std,normalised_improvement_mean,normalised_improvement_std,max_IoU_mean,max_IoU_std,steps_mean,steps_std,num_auto_success_nc,num_success_sum,num_trials_sum,agent
0,94.192542,7.332183,88.663872,14.279365,83.261533,13.946307,10.775000,5.595270,33,31,40,Human Policy
0,83.097266,11.687125,68.521719,19.350266,71.345811,8.845085,18.666667,3.862582,9,4,30,LaGarNet
0,68.699252,13.721316,39.446618,26.059435,63.191946,12.261898,19.125000,3.106548,3,3,40,PlaNet-ClothPick


,fabric,normalised_coverage_mean,normalised_coverage_std,normalised_improvement_mean,normalised_improvement_std,max_IoU_mean,max_IoU_std,steps_mean,steps_std,num_auto_success,num_auto_success_nc,num_trials,agent
0,Dress,86.831287,8.745900,73.801409,16.791307,76.658806,9.765602,15.9,5.839521,4,5,10,Human Policy
1,Longsleeve,97.285456,3.133560,95.001110,5.476799,79.632265,23.398560,10.6,3.893014,9,10,10,Human Policy
2,Trousers,96.313860,6.779995,93.385394,12.155302,87.307250,7.606221,10.7,4.989990,9,9,10,Human Policy
3,Skirt,96.339564,4.441368,92.467575,9.642009,89.447811,4.281707,5.9,2.424413,9,9,10,Human Policy
0,Dress,76.816847,9.945994,58.261227,16.466097,66.288907,7.680274,20.0,0.000000,0,0,10,LaGarNet
1,Longsleeve,89.892942,13.404267,81.205385,21.530813,73.709774,10.169911,17.7,4.001389,3,8,10,LaGarNet
2,Trousers,82.582008,8.129504,66.098544,13.021047,74.038751,6.855508,18.3,5.375872,1,1,10,LaGarNet
0,Dress,60.596332,12.904580,26.822321,17.877330,55.284350,10.534065,20.0,0.000000,0,0,10,PlaNet-ClothPick
1,Longsleeve,64.942419,9.906918,29.025099,24.311564,59.669784,6.327134,20.0,0.000000,0,0,10,PlaNet-ClothPick
2,Trousers,72.181721,13.007406,46.789841,23.572955,63.255696,11.553506,19.9,0.316228,0,0,10,PlaNet-ClothPick


In [5]:
import pandas as pd

# Define the garment (fabric) order for the table
garments = ['Longsleeve', 'Trousers', 'Dress', 'Skirt']

# Define the agents in the order for columns
agents = ['PlaNet-ClothPick', 'LaGarNet', 'Human Policy']

# Helper function to format mean ± std
def fmt(mean, std, decimals=1):
    if pd.isnull(mean) or pd.isnull(std):
        return ''
    return f"{mean:.{decimals}f} $\\pm$ {std:.{decimals}f}"

# Helper function for SR (success rate)
def fmt_sr(success, trials):
    if pd.isnull(success):
        return f"{0}/{int(trials)}"
    return f"{int(success)}/{int(trials)}"

# Build a dict to easily access each agent's data by fabric
def get_agent_fabric_stats(df, agent, fabric):
    row = df[(df['agent'] == agent) & (df['fabric'] == fabric)]
    #display(row)
    if row.empty:
        return {}
    row = row.iloc[0]
    return {
        'NC': fmt(row['normalised_coverage_mean'], row['normalised_coverage_std']),
        'NI': fmt(row['normalised_improvement_mean'], row['normalised_improvement_std']),
        'Max IoU': fmt(row['max_IoU_mean'], row['max_IoU_std']),
        'SR^{90}_{80}': fmt_sr(row['num_auto_success'], row['num_trials']),
        'SR^{90}_{0}': fmt_sr(row['num_auto_success_nc'], row['num_trials']),
        'S2F': fmt(row['steps_mean'], row['steps_std'])
    }

# Prepare LaTeX table header
latex = r"""\begin{tabular}{c|cccccc|cccccc||cccccc}
\hline
Method & \multicolumn{6}{c|}{PlaNet-ClothPick \cite{kadi2024planet}}  & \multicolumn{6}{c||}{LaGarNet} & \multicolumn{6}{c}{Human Policy} \\
\cline{1-19}
Garments 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ \\
\hline
"""



# Fill in the table rows
for fabric in garments:
    row = [fabric]
    # For each agent, get stats for this fabric
    for agent in agents:
        stats = get_agent_fabric_stats(total_objects_states, agent, fabric)
        #print('stats', stats)
        row.extend([stats.get('NC', ''), stats.get('NI', ''), stats.get('Max IoU', ''), stats.get('S2F', ''), stats.get('SR^{90}_{80}', ''), stats.get('SR^{90}_{0}', '')])
    latex += ' & '.join(row) + r' \\' + '\n'

latex += r"\hline" + '\n'

def get_total_stats(df, agent):
    rows = df[df['agent'] == agent]
    if rows.empty:
        return {}
    # Weighted mean and std for metrics
    def weighted_mean_std(mean_col, std_col, count_col):
        means = rows[mean_col]
        stds = rows[std_col]
        counts = rows[count_col]
        total = counts.sum()
        if total == 0:
            return float('nan'), float('nan')
        weighted_mean = (means * counts).sum() / total
        # Pooled std (approximate)
        weighted_std = (stds * counts).sum() / total
        return weighted_mean, weighted_std

    # For NC, NI, Max IoU, S2F
    nc_mean, nc_std = weighted_mean_std('normalised_coverage_mean', 'normalised_coverage_std', 'num_trials_sum')
    ni_mean, ni_std = weighted_mean_std('normalised_improvement_mean', 'normalised_improvement_std', 'num_trials_sum')
    iou_mean, iou_std = weighted_mean_std('max_IoU_mean', 'max_IoU_std', 'num_trials_sum')
    s2f_mean, s2f_std = weighted_mean_std('steps_mean', 'steps_std', 'num_trials_sum')
    # For SR
    sr_success = rows['num_success_sum'].sum()
    sr_trials = rows['num_trials_sum'].sum()
    sr_nc_success = rows['num_auto_success_nc'].sum()
    return {
        'NC': fmt(nc_mean, nc_std),
        'NI': fmt(ni_mean, ni_std),
        'Max IoU': fmt(iou_mean, iou_std),
        'SR^{90}_{80}': fmt_sr(sr_success, sr_trials),
        'SR^{90}_{0}': fmt_sr(sr_nc_success, sr_trials),
        'S2F': fmt(s2f_mean, s2f_std)
    }

row = ['Real Total']
for agent in agents:
    stats = get_total_stats(total_states, agent)
    row.extend([stats.get('NC', ''), stats.get('NI', ''), stats.get('Max IoU', ''), stats.get('S2F', ''), stats.get('SR^{90}_{80}', ''), stats.get('SR^{90}_{0}', '')])
latex += ' & '.join(row) + r' \\' + '\n'

latex += r"""
\end{tabular}
"""

print(latex)


\begin{tabular}{c|cccccc|cccccc||cccccc}
\hline
Method & \multicolumn{6}{c|}{PlaNet-ClothPick \cite{kadi2024planet}}  & \multicolumn{6}{c||}{LaGarNet} & \multicolumn{6}{c}{Human Policy} \\
\cline{1-19}
Garments 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$   & S2F $\downarrow$  & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ \\
\hline
Longsleeve & 64.9 $\pm$ 9.9 & 29.0 $\pm$ 24.3 & 59.7 $\pm$ 6.3 & 20.0 $\pm$ 0.0 & 0/10 & 0/10 & 89.9 $\pm$ 13.4 & 81.2 $\pm$ 21.5 & 73.7 $\pm$ 10.2 & 17.7 $\pm$ 4.0 & 3/10 & 8/10 & 97.3 $\pm$ 3.1 & 95.0 $\pm$ 5.5 & 79.6 $\pm$ 23.4 & 10.6 $\pm$ 3.9 & 9/10 & 10/10 \\
Trousers & 72.2 $\pm$ 13.0 & 46.8 $\pm$ 23.6 & 63.3 $\pm$ 11.6 & 19.9 $\pm$ 0.3 & 0/10 & 0/10 & 82.6 $\pm$ 8